# Social Sensing: análisis del discurso político de Donald Trump

**Sistemas Inteligentes · Universitat Politècnica de València (UPV)**  
**Duración estimada: 60–90 minutos · Google Colab · CPU**

El **Social Sensing** transforma publicaciones digitales en señales estructuradas para estudiar fenómenos sociales y políticos. Las redes permiten observar temas, emociones y referencias a instituciones, pero sus mensajes no son un sondeo representativo ni una medición directa de intenciones.

**Objetivos de aprendizaje**
1. Cargar y explorar mensajes de una red social.
2. Definir una taxonomía temática.
3. Clasificar mensajes mediante técnicas sencillas de PLN.
4. Aplicar análisis de sentimiento.
5. Analizar la relación entre temática, sentimiento y volumen.
6. Reflexionar sobre los límites del análisis automático.

Estudiaremos **tema**, **sentimiento** y **postura hacia un objetivo institucional** como dimensiones diferentes. Una crítica política, incluso muy negativa, no equivale a un ataque a la democracia. Las conclusiones deben apoyarse en los datos y en la revisión humana; no las fijamos de antemano.

Los textos pueden contener insultos y afirmaciones controvertidas. Las citas son material de análisis, no hechos verificados ni posiciones del curso.

## 0. Clonar el repositorio e instalar dependencias

La primera celda ejecutable clona el repositorio con `git clone https://github.com/chemacecilia/SII-UPV.git` cuando no hay una copia disponible. En Colab se crea `SII-UPV/` en el directorio de trabajo y se carga `SII-UPV/datos/trump_dataset.csv`.

Si abriste el notebook desde la raíz del repositorio o desde `codigo/`, reutilizamos esa copia para evitar clones anidados. Si existe una carpeta `SII-UPV` ajena al repositorio, detenemos la ejecución sin sobrescribirla. La clonación inicial y la instalación necesitan Internet; el análisis posterior es local y no requiere claves ni modelos grandes.

In [ ]:
from pathlib import Path
import subprocess

URL_REPO = 'https://github.com/chemacecilia/SII-UPV.git'
candidatas = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'SII-UPV']
RUTA_REPO = None
for candidata in candidatas:
    if (candidata / '.git').exists() and (candidata / 'datos/trump_dataset.csv').is_file():
        origen = subprocess.run(['git', '-C', str(candidata), 'remote', 'get-url', 'origin'],
                                check=True, capture_output=True, text=True).stdout.strip()
        if origen.rstrip('/').removesuffix('.git') == URL_REPO.removesuffix('.git'):
            RUTA_REPO = candidata.resolve()
            break

if RUTA_REPO is None:
    destino = Path.cwd() / 'SII-UPV'
    if destino.exists():
        raise FileExistsError(f'{destino} ya existe y no es la copia esperada. Revísala antes de continuar.')
    subprocess.run(['git', 'clone', URL_REPO], check=True)
    RUTA_REPO = destino.resolve()

RUTA_DATOS = RUTA_REPO / 'datos' / 'trump_dataset.csv'
if not RUTA_DATOS.is_file():
    raise FileNotFoundError(RUTA_DATOS)
revision_repo = subprocess.run(['git', '-C', str(RUTA_REPO), 'rev-parse', 'HEAD'],
                               check=True, capture_output=True, text=True).stdout.strip()
print('CSV:', RUTA_DATOS)
print('Revisión de los datos:', revision_repo)

In [ ]:
import sys
import importlib.metadata as metadata

# VADER incluye su léxico: no necesitamos nltk.download ni acceso a APIs.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'vaderSentiment==3.3.2',
                'pandas>=2.2,<3', 'numpy>=1.26,<3',
                'matplotlib>=3.8,<4', 'seaborn==0.13.2'], check=True)

import re
import unicodedata
from html.parser import HTMLParser
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

SEMILLA = 42
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})
PALETA = {'Positivo': '#268577', 'Neutro': '#87939e', 'Negativo': '#bb505c'}
ORDEN_SENTIMIENTO = ['Positivo', 'Neutro', 'Negativo']
print({p: metadata.version(p) for p in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'vaderSentiment']})

## 1. Inspeccionar el fichero real

Este CSV no tiene el esquema del fichero de Torrevieja: usa **punto y coma**, codificación **Windows-1252** y las columnas `date`, `favorites`, `id`, `isRetweet`, `retweets`, `text`. La columna de mensajes es `text`; `id` se conserva como cadena para evitar pérdida de precisión.

Primero mostramos el esquema y los registros sin transformarlos. El esquema se ha comprobado con el fichero del repositorio; si cambia, una comprobación explícita evita analizar una columna equivocada.

In [ ]:
df_original = pd.read_csv(RUTA_DATOS, sep=';', encoding='cp1252', dtype='string')
COLUMNAS_ESPERADAS = {'date', 'favorites', 'id', 'isRetweet', 'retweets', 'text'}
if not COLUMNAS_ESPERADAS.issubset(df_original.columns):
    raise ValueError(f'El esquema ha cambiado. Columnas disponibles: {df_original.columns.tolist()}')
COLUMNA_TEXTO = 'text'
print('Número total de registros:', len(df_original))
print('Columnas disponibles:', df_original.columns.tolist())
display(df_original.head(5))
display(df_original.isna().sum().rename('valores_nulos').to_frame())
print('Filas completamente duplicadas:', int(df_original.duplicated().sum()))
print('IDs repetidos no nulos:', int(df_original.loc[df_original['id'].notna(), 'id'].duplicated().sum()))
print('Textos exactamente repetidos:', int(df_original[COLUMNA_TEXTO].duplicated().sum()))

### HTML, mensajes vacíos y duplicados

Algunos registros incorporan etiquetas HTML procedentes de la exportación. Creamos `texto_original` sin cambios y `texto_legible` sin etiquetas, conservando contenido, mayúsculas, negaciones y puntuación.

Eliminamos textos vacíos e IDs repetidos (conservando la primera aparición). Un texto repetido con otro ID se conserva: puede ser una publicación distinta. Mostramos cuántos hay para discutir cómo afectarían a los resultados. No interpretamos automáticamente los retuits como texto original ni como adhesión personal.

In [ ]:
class ExtractorTexto(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.partes = []

    def handle_starttag(self, tag, attrs):
        if tag in {'p', 'br', 'div', 'li'}:
            self.partes.append(' ')

    def handle_endtag(self, tag):
        if tag in {'p', 'div', 'li'}:
            self.partes.append(' ')

    def handle_data(self, data):
        self.partes.append(data)


def quitar_html(text):
    if pd.isna(text):
        return ''
    parser = ExtractorTexto()
    parser.feed(str(text))
    return re.sub(r'\s+', ' ', ''.join(parser.partes)).strip()


df = df_original.copy()
df['texto_original'] = df[COLUMNA_TEXTO]
df['texto_legible'] = df['texto_original'].map(quitar_html)
vacios = df['texto_legible'].eq('')
print('Mensajes vacíos después de retirar HTML:', int(vacios.sum()))
df = df.loc[~vacios].copy()
ids_repetidos = df['id'].notna() & df['id'].duplicated()
print('Registros descartados por ID repetido:', int(ids_repetidos.sum()))
df = df.loc[~ids_repetidos].reset_index(drop=True)
if df.empty:
    raise ValueError('No hay mensajes no vacíos para analizar.')
df.index.name = 'fila'
print('Mensajes procesables:', len(df))
print('Textos legibles repetidos conservados:', int(df['texto_legible'].duplicated().sum()))
df['es_retuit'] = df['isRetweet'].str.upper().map({'VERDADERO': True, 'FALSO': False}).astype('boolean')
print('Indicador de retuit (incluye desconocidos):')
display(df['es_retuit'].value_counts(dropna=False).rename('mensajes').to_frame())
for col in ['favorites', 'retweets']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

### Fechas y cobertura: comprobar antes de interpretar

`date` contiene marcas de tiempo Unix en **milisegundos**, expresadas en notación científica con coma decimal (`1,73039E+12`). Convertimos la coma a punto y después a fecha UTC; las fechas no interpretables se mantienen como `NaT` y se excluyen solo de los gráficos temporales.

La exportación redondea esas marcas: **no inferimos la hora exacta** ni analizamos minutos. Agregamos por meses. Los huecos temporales y los cambios de volumen pueden deberse a la selección del corpus y no a la actividad real de la cuenta.

In [ ]:
fecha_numerica = pd.to_numeric(df['date'].str.replace(',', '.', regex=False), errors='coerce')
df['fecha'] = pd.to_datetime(fecha_numerica, unit='ms', origin='unix', utc=True, errors='coerce')
print('Fechas válidas:', int(df['fecha'].notna().sum()), '/', len(df))
print('Cobertura observada (UTC):', df['fecha'].min(), '—', df['fecha'].max())
display(df[['date', 'fecha']].head())
df['longitud'] = df['texto_legible'].str.len()
print(f"Longitud media: {df['longitud'].mean():.1f} caracteres (sin HTML)")
print(f"Mediana: {df['longitud'].median():.0f} caracteres")
sns.histplot(data=df, x='longitud', bins=35, color='#3677a5')
plt.title('Longitud de los mensajes del corpus')
plt.xlabel('Caracteres del texto legible')
plt.ylabel('Mensajes')
plt.tight_layout()
plt.show()

fechas_validas = df['fecha'].dropna()
HAY_TIEMPO = len(fechas_validas) >= 10 and fechas_validas.dt.strftime('%Y-%m').nunique() >= 2
if HAY_TIEMPO:
    volumen_temporal = (df.dropna(subset=['fecha']).set_index('fecha')
                        .resample('MS').size().rename('mensajes'))
    volumen_temporal.plot(color='#3677a5')
    plt.title('Mensajes presentes en el corpus por mes (no actividad total de la cuenta)')
    plt.xlabel('Mes (UTC)')
    plt.ylabel('Mensajes')
    plt.tight_layout()
    plt.show()
else:
    print('Se omite el gráfico temporal: se requieren al menos 10 fechas válidas en 2 meses distintos.')

In [ ]:
contiene_fake_news = df['texto_legible'].str.contains(r'\bfake\s+news\b', case=False, regex=True)
print(f'Mensajes con «fake news»: {contiene_fake_news.sum()} de {len(df)} '
      f'({100 * contiene_fake_news.mean():.1f} %)')
# Diagnóstico descriptivo: no sustituye a la documentación de la recogida del dataset.
display(df[['id', 'texto_legible']].sample(min(3, len(df)), random_state=SEMILLA))

**Advertencia sobre la muestra.** La fuerte presencia de «fake news» es compatible con una selección centrada en ese término; el CSV no documenta cómo se recogió. No afirmamos conocer el mecanismo de selección. Los resultados describen **este fichero**, no todos los mensajes publicados por Trump, la opinión pública ni la evolución completa de su discurso. Tampoco existe una columna que permita verificar la autoría de cada texto citado o retuiteado.

## 2. Preprocesamiento: conservar las señales relevantes

Para buscar temas, pasamos a minúsculas, retiramos URLs y menciones, quitamos el símbolo `#` conservando la palabra del hashtag y normalizamos espacios y comillas. No quitamos negaciones como **not, never, no**, ni eliminamos palabras vacías de forma general.

Para **VADER** usaremos otra representación que conserva mayúsculas y puntuación: «GREAT!!!» no aporta las mismas señales que «great». Conservamos siempre el original para volver al contexto.

In [ ]:
PATRON_URL = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
PATRON_MENCION = re.compile(r'(?<!\w)@\w+')


def preprocess_text(text):
    texto = unicodedata.normalize('NFKC', quitar_html(text)).lower()
    texto = texto.replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"')
    texto = PATRON_URL.sub(' ', texto)
    texto = PATRON_MENCION.sub(' ', texto)
    texto = re.sub(r'#(?=\w)', '', texto)
    texto = re.sub(r'[\x00-\x1f\x7f\u200b\ufeff]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def preparar_vader(text):
    texto = quitar_html(text)
    texto = PATRON_URL.sub(' ', texto)
    texto = PATRON_MENCION.sub(' ', texto)
    texto = re.sub(r'#(?=\w)', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df['texto_procesado'] = df['texto_original'].map(preprocess_text)
df['texto_sentimiento'] = df['texto_original'].map(preparar_vader)
display(df[['texto_original', 'texto_procesado']].head(3))
assert all(p in preprocess_text('Not never no #Election') for p in ['not', 'never', 'no', 'election'])

## 3. Definir una taxonomía democrática

El diccionario es una **operacionalización didáctica**, no una definición exhaustiva de democracia ni un detector de ataques. Incluye instituciones, procedimientos, libertades y actores políticos en inglés, con expresiones y variantes. Edita las listas y vuelve a ejecutar la clasificación para observar su sensibilidad.

Algunas señales son ambiguas: `administration` puede usarse fuera de la política, `right` puede significar “correcto” y una mención a una persona no siempre identifica una institución. Por eso evitamos términos demasiado generales como `right` o `state`, aunque la cobertura siga siendo imperfecta. Las categorías pueden solaparse.

In [ ]:
TAXONOMIA = {
    'Instituciones públicas': [
        'government', 'federal government', 'administration', 'congress', 'congressional',
        'senate', 'senator', 'senators', 'house of representatives', 'white house',
        'federal agency', 'federal agencies', 'federal official', 'federal officials',
        'fbi', 'cia', 'public officials', 'civil service'],
    'Sistema judicial': [
        'judge', 'judges', 'court', 'courts', 'supreme court', 'supreme court justices',
        'justice department', 'department of justice', 'doj', 'prosecutor', 'prosecutors',
        'attorney general', 'district attorney', 'rule of law', 'judiciary', 'judicial'],
    'Prensa y libertad de información': [
        'press', 'free press', 'freedom of the press', 'freedom of information',
        'media', 'mainstream media', 'news', 'fake news', 'journalist', 'journalists',
        'reporter', 'reporters', 'cnn', 'msnbc', 'fox news', 'new york times',
        'washington post', 'cbs', 'abc', 'disinformation', 'misinformation'],
    'Elecciones y sistema electoral': [
        'election', 'elections', 'electoral', 'vote', 'votes', 'voter', 'voters',
        'voting', 'ballot', 'ballots', 'mail-in voting', 'mail in ballots',
        'vote count', 'recount', 'election results', 'polling place', 'electoral college'],
    'Separación de poderes y controles democráticos': [
        'checks and balances', 'separation of powers', 'congressional oversight',
        'oversight', 'impeachment', 'subpoena', 'subpoenas', 'special counsel',
        'congressional investigation', 'congressional investigations',
        'judicial independence', 'executive power', 'executive order', 'executive orders'],
    'Derechos y libertades civiles': [
        'civil rights', 'civil liberties', 'constitutional rights', 'free speech',
        'freedom of speech', 'first amendment', 'equal protection', 'equal rights',
        'due process', 'right to protest', 'protest', 'protests', 'peaceful assembly',
        'voting rights', 'citizen participation'],
    'Oposición y pluralismo político': [
        'democrat', 'democrats', 'democratic party', 'republican', 'republicans',
        'republican party', 'political opponent', 'political opponents', 'opposition',
        'bipartisan', 'bipartisanship', 'pluralism', 'political parties',
        'biden', 'harris', 'kamala', 'clinton', 'pelosi', 'schumer', 'bernie sanders'],
    'Otros': [],
}
# Este orden también fija cómo se resuelven empates de la temática principal.
CATEGORIAS = list(TAXONOMIA)
print('Categorías:', *CATEGORIAS, sep='\n- ')

## 4. Clasificación temática multietiqueta

`classify_topics(text)` devuelve todas las categorías con al menos una coincidencia. Usamos límites de palabra para evitar, por ejemplo, que `court` coincida con una parte de otra palabra. Se aceptan expresiones con varios espacios.

Para `primary_topic` contamos **expresiones distintas** encontradas en cada categoría, sin premiar la repetición del mismo término. Gana la categoría con más expresiones; en empate, la que aparece antes en el diccionario. Expresiones anidadas como `court` y `supreme court` pueden sumar por separado: es una regla transparente, no una puntuación semántica validada. `Otros` se asigna solo si no hay coincidencias.

**Importante:** detectar el nombre de una institución no determina si el mensaje la elogia, critica, ataca o deslegitima.

In [ ]:
def compilar_taxonomia(taxonomia):
    return {
        categoria: [re.compile(r'(?<!\w)' + re.escape(expresion.lower()).replace(r'\ ', r'\s+') + r'(?!\w)')
                    for expresion in dict.fromkeys(expresiones)]
        for categoria, expresiones in taxonomia.items() if categoria != 'Otros'
    }


PATRONES_TEMAS = compilar_taxonomia(TAXONOMIA)


def coincidencias_tematicas(text):
    texto = preprocess_text(text)
    return {categoria: sum(bool(patron.search(texto)) for patron in patrones)
            for categoria, patrones in PATRONES_TEMAS.items()}


def classify_topics(text):
    coincidencias = coincidencias_tematicas(text)
    return [categoria for categoria, cantidad in coincidencias.items() if cantidad > 0] or ['Otros']


def tema_principal(text):
    coincidencias = coincidencias_tematicas(text)
    if not any(coincidencias.values()):
        return 'Otros'
    return max(coincidencias, key=coincidencias.get)


df['topics'] = df['texto_procesado'].map(classify_topics)
# Otros es una etiqueta de reserva, no un tema sustantivo reconocido.
df['n_topics'] = df['topics'].map(lambda temas: 0 if temas == ['Otros'] else len(temas))
df['primary_topic'] = df['texto_procesado'].map(tema_principal)
display(df[['id', 'texto_legible', 'topics', 'n_topics', 'primary_topic']].head(5))
print('Mensajes sin coincidencias (Otros):', int(df['n_topics'].eq(0).sum()))
print('Mensajes con varias temáticas:', int(df['n_topics'].gt(1).sum()))

In [ ]:
# Una fila por pareja mensaje-tema; cada mensaje cuenta una vez en cada categoría asignada.
df_temas = df[['topics']].explode('topics').rename(columns={'topics': 'topic'})
resumen_temas = df_temas['topic'].value_counts().reindex(CATEGORIAS, fill_value=0).rename('mensajes').to_frame()
resumen_temas['porcentaje'] = (100 * resumen_temas['mensajes'] / len(df)).round(2)
display(resumen_temas)
print('Denominador:', len(df), 'mensajes; los porcentajes pueden sumar más del 100 % por el solapamiento.')

ax = resumen_temas['mensajes'].sort_values().plot.barh(color='#3677a5', figsize=(11, 6))
ax.set_title('Gráfico 1 · Mensajes por categoría temática (multietiqueta)')
ax.set_xlabel('Número de mensajes con al menos una coincidencia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

# Un ejemplo reproducible por categoría para revisar los aciertos y errores.
for categoria in CATEGORIAS:
    candidatas = df.loc[df['topics'].map(lambda temas: categoria in temas)]
    print('\nCategoría:', categoria, '| mensajes:', len(candidatas))
    if not candidatas.empty:
        ejemplo = candidatas.sample(1, random_state=SEMILLA).iloc[0]
        print('ID:', ejemplo['id'], '|', ejemplo['texto_legible'])

## 5. Sentimiento con VADER

VADER es ligero, no requiere entrenamiento y constituye un punto de partida para textos breves en inglés. Su léxico y reglas usan negaciones, intensidad, puntuación y mayúsculas. No necesita modelos de gran tamaño ni servicios de pago.

Calculamos `negative`, `neutral`, `positive` y `compound`. Los tres primeros son proporciones internas de VADER, no probabilidades calibradas de acertar. `compound` combina el tono en una escala de −1 a +1.

Umbrales habituales: **Positivo** si `compound >= 0.05`; **Negativo** si `compound <= -0.05`; **Neutro** en el intervalo abierto intermedio. El sentimiento describe tono emocional; no mide veracidad, objetivo de la emoción ni orientación democrática.

In [ ]:
analizador = SentimentIntensityAnalyzer()
puntuaciones = pd.DataFrame(df['texto_sentimiento'].map(analizador.polarity_scores).tolist(), index=df.index)
puntuaciones = puntuaciones.rename(columns={'neg': 'negative', 'neu': 'neutral', 'pos': 'positive'})
# Permite repetir esta celda sin duplicar columnas.
df = df.drop(columns=['negative', 'neutral', 'positive', 'compound'], errors='ignore').join(puntuaciones)
df['sentiment_score'] = df['compound']
df['sentiment_label'] = np.select(
    [df['sentiment_score'].ge(0.05), df['sentiment_score'].le(-0.05)],
    ['Positivo', 'Negativo'], default='Neutro')

display(df[['texto_legible', 'negative', 'neutral', 'positive', 'compound', 'sentiment_label']].head())
for etiqueta in ORDEN_SENTIMIENTO:
    candidatas = df.loc[df['sentiment_label'].eq(etiqueta)]
    print('\nSentimiento VADER:', etiqueta, '| mensajes:', len(candidatas))
    if not candidatas.empty:
        display(candidatas.sample(min(2, len(candidatas)), random_state=SEMILLA)
                [['id', 'texto_legible', 'sentiment_score']])

resumen_sentimiento = df['sentiment_label'].value_counts().reindex(ORDEN_SENTIMIENTO, fill_value=0)
display(resumen_sentimiento.rename('mensajes').to_frame().assign(
    porcentaje=lambda tabla: (100 * tabla['mensajes'] / len(df)).round(2)))
resumen_sentimiento.plot.bar(color=[PALETA[s] for s in ORDEN_SENTIMIENTO], rot=0)
plt.title('Gráfico 2 · Distribución del sentimiento según VADER')
plt.xlabel('Sentimiento automático')
plt.ylabel('Mensajes')
plt.tight_layout()
plt.show()

## 6. Postura hacia las instituciones: revisión humana

**Sentimiento y postura no son intercambiables.** Un tono positivo puede acompañar una propuesta que niegue legitimidad a un proceso, y un tono negativo puede expresar una crítica legítima a una decisión concreta. Además, el tono global puede dirigirse a una persona o hecho ajeno a la institución mencionada.

Anotaremos **un objetivo explícito por fila**. Si hay varios, se necesitan anotaciones separadas; esta ficha simplificada solo registra el principal. La rúbrica propone cuatro etiquetas:

| Etiqueta | Criterio de anotación | Qué no basta |
|---|---|---|
| Neutral / Descriptive | Describe o menciona el objetivo sin evaluación explícita. | La mera coincidencia de una palabra no prueba una mención institucional. |
| Legitimate political criticism | Critica una decisión, actuación o política concreta sin un ataque personal ni una negación explícita de legitimidad. “Legitimate” describe aquí la forma de la crítica, no certifica que la acusación sea cierta. | Negatividad, desacuerdo o pedir una reforma. |
| Institutional or personal attack | Insulta, denigra o desacredita de manera general a una persona o institución identificable. | Una crítica específica y argumentada, por dura que sea. |
| Explicit institutional delegitimization | Niega explícitamente la legitimidad de una institución o procedimiento, o pide desconocer su autoridad o resultados fuera de los mecanismos previstos. Debe citarse evidencia textual y revisarse su contexto. | Una alegación aislada de error, una investigación, un recurso judicial o el sentimiento negativo. |

La rúbrica de cuatro etiquetas no cubre todas las posturas posibles: por ejemplo, el elogio no es neutral. Para mensajes favorables, ambiguos o sin objetivo identificable, **deja la postura pendiente y explica el motivo**, sin forzar una etiqueta. En una ampliación convendría añadir una etiqueta favorable. No tratamos estas etiquetas como una determinación jurídica ni como inferencias automáticas de intención antidemocrática.

Dos personas deberían anotar independientemente y discutir desacuerdos con el texto completo y, cuando sea posible, contexto externo. La última etiqueta exige especial prudencia.

In [ ]:
# Muestra aleatoria para revisión. No es un conjunto de etiquetas ya validadas.
columnas_muestra = ['id', 'texto_legible', 'topics', 'sentiment_label']
muestra_postura = df.sample(min(20, len(df)), random_state=SEMILLA)[columnas_muestra].copy()
muestra_postura['objetivo'] = ''
muestra_postura['postura_manual'] = pd.Series(pd.NA, index=muestra_postura.index, dtype='string')
muestra_postura['evidencia_textual'] = ''
muestra_postura['justificacion'] = ''
muestra_postura['anotador'] = ''
display(muestra_postura)

# Heurística SOLO para localizar candidatos adicionales: no asigna postura.
# “fake news”, por ejemplo, puede aparecer citado o sin un objetivo institucional concreto.
PATRON_CANDIDATOS = r'\b(?:fake news|enemy of the people|rigged|stolen election|illegitimate|witch hunt)\b'
candidatos_revision = df.loc[df['texto_procesado'].str.contains(PATRON_CANDIDATOS, regex=True, na=False)]
print('Candidatos por coincidencia léxica (NO ataques confirmados):', len(candidatos_revision))
display(candidatos_revision[columnas_muestra].sample(min(5, len(candidatos_revision)), random_state=SEMILLA))

In [ ]:
ETIQUETAS_POSTURA = [
    'Neutral / Descriptive',
    'Legitimate political criticism',
    'Institutional or personal attack',
    'Explicit institutional delegitimization',
]


def anotar_postura(fila, etiqueta, objetivo, evidencia, justificacion, anotador):
    """Anotar una fila de la muestra; requiere un fragmento literal y una justificación."""
    if fila not in muestra_postura.index:
        raise KeyError('Selecciona el índice de una fila de muestra_postura.')
    if etiqueta not in ETIQUETAS_POSTURA:
        raise ValueError(f'Etiquetas permitidas: {ETIQUETAS_POSTURA}')
    if not all(isinstance(x, str) and x.strip() for x in [objetivo, evidencia, justificacion, anotador]):
        raise ValueError('Completa objetivo, evidencia literal, justificación y anotador.')
    if evidencia not in muestra_postura.at[fila, 'texto_legible']:
        raise ValueError('La evidencia debe ser un fragmento literal del texto legible.')
    muestra_postura.loc[fila, ['objetivo', 'postura_manual', 'evidencia_textual', 'justificacion', 'anotador']] = [
        objetivo, etiqueta, evidencia, justificacion, anotador]


# Tras leer el texto completo, llama a la función con el índice mostrado en la tabla:
# anotar_postura(fila=..., etiqueta=ETIQUETAS_POSTURA[...], objetivo='...',
#               evidencia='fragmento literal', justificacion='criterio y dudas', anotador='grupo 1')
print('Anotaciones realizadas:', int(muestra_postura['postura_manual'].notna().sum()))
print('No se calculan tasas de ataques: faltan anotaciones contrastadas y una evaluación de fiabilidad.')
# Exportación opcional (evita sobrescribir el dataset original):
# muestra_postura.to_csv('anotaciones_postura_grupo.csv', index=True, encoding='utf-8')

## 7. Relacionar volumen, tema y sentimiento

Para las siguientes tablas, un mensaje que menciona dos categorías contribuye **una vez a cada una**. Por eso los recuentos temáticos no son una partición del corpus. El porcentaje de sentimiento se calcula **dentro de cada categoría**, no respecto al total de asignaciones.

Una categoría con cero mensajes no tiene una distribución estimable; su fila se deja sin porcentaje en el mapa de calor. Los gráficos no cuantifican ataques ni postura institucional.

In [ ]:
df_temas_sentimiento = (df[['topics', 'sentiment_label', 'sentiment_score', 'fecha']]
                        .explode('topics').rename(columns={'topics': 'topic'}))
tabla_conjunta = pd.crosstab(df_temas_sentimiento['topic'], df_temas_sentimiento['sentiment_label'])
tabla_conjunta = tabla_conjunta.reindex(index=CATEGORIAS, columns=ORDEN_SENTIMIENTO, fill_value=0)
porcentajes_categoria = tabla_conjunta.div(tabla_conjunta.sum(axis=1).replace(0, np.nan), axis=0) * 100
display(tabla_conjunta)
display(porcentajes_categoria.round(1))

# Etiquetas cortas solo para mejorar la legibilidad de los gráficos.
NOMBRES_CORTOS = {
    'Instituciones públicas': 'Instituciones\npúblicas',
    'Sistema judicial': 'Sistema\njudicial',
    'Prensa y libertad de información': 'Prensa e\ninformación',
    'Elecciones y sistema electoral': 'Sistema\nelectoral',
    'Separación de poderes y controles democráticos': 'Controles\ndemocráticos',
    'Derechos y libertades civiles': 'Derechos\ny libertades',
    'Oposición y pluralismo político': 'Oposición y\npluralismo',
    'Otros': 'Otros',
}
ax = tabla_conjunta.rename(index=NOMBRES_CORTOS).plot.bar(
    stacked=True, color=[PALETA[s] for s in ORDEN_SENTIMIENTO], figsize=(12, 6), rot=0, width=0.75)
ax.set_title('Gráfico 3 · Volumen por temática y sentimiento (multietiqueta)')
ax.set_xlabel('Categoría temática')
ax.set_ylabel('Mensajes')
ax.legend(title='Sentimiento VADER', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(porcentajes_categoria.rename(index=NOMBRES_CORTOS), annot=True, fmt='.1f',
            cmap='Blues', vmin=0, vmax=100, linewidths=0.5,
            mask=porcentajes_categoria.isna().to_numpy(), cbar_kws={'label': '% dentro de la categoría'})
plt.title('Gráfico 4 · Distribución porcentual del sentimiento en cada categoría')
plt.xlabel('Sentimiento VADER')
plt.ylabel('Categoría temática')
plt.tight_layout()
plt.show()

In [ ]:
if HAY_TIEMPO:
    temporales = df.dropna(subset=['fecha']).set_index('fecha').sort_index()
    temporal = temporales.resample('MS').agg(
        mensajes=('sentiment_score', 'size'), media_compound=('sentiment_score', 'mean'))
    conteos_tiempo = pd.crosstab(temporales.index.strftime('%Y-%m'), temporales['sentiment_label'])
    conteos_tiempo = conteos_tiempo.reindex(columns=ORDEN_SENTIMIENTO, fill_value=0)
    conteos_tiempo.index = pd.to_datetime(conteos_tiempo.index, format='%Y-%m', utc=True)
    conteos_tiempo = conteos_tiempo.reindex(temporal.index, fill_value=0)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for etiqueta in ORDEN_SENTIMIENTO:
        axes[0].plot(conteos_tiempo.index, conteos_tiempo[etiqueta],
                     label=etiqueta, color=PALETA[etiqueta], linewidth=1.5)
    axes[0].set_title('Gráfico 5 · Evolución de los mensajes presentes en el corpus')
    axes[0].set_ylabel('Mensajes por mes')
    axes[0].legend(title='VADER', loc='upper right')
    # Media mensual solo con al menos cinco mensajes: reduce inestabilidad, no elimina sesgos.
    media_visible = temporal['media_compound'].where(temporal['mensajes'].ge(5))
    axes[1].plot(temporal.index, media_visible, color='#7354a3', marker='.', label='Media con n ≥ 5')
    axes[1].axhline(0, color='gray', linewidth=1, linestyle='--')
    axes[1].set_ylim(-1, 1)
    axes[1].set_ylabel('Compound medio')
    axes[1].set_xlabel('Mes (UTC); huecos en la media = menos de 5 mensajes')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    # Tabla de temas por mes: permite buscar cambios sin confundirlos con sentimiento o ataques.
    temas_temporales = df_temas_sentimiento.dropna(subset=['fecha']).copy()
    temas_temporales['mes'] = temas_temporales['fecha'].dt.strftime('%Y-%m')
    volumen_temas_mes = pd.crosstab(temas_temporales['mes'], temas_temporales['topic']).reindex(
        columns=CATEGORIAS, fill_value=0)
    print('Volumen multietiqueta por tema en los últimos meses observados:')
    display(volumen_temas_mes.tail(12))
else:
    print('Gráfico 5 omitido: no hay suficientes fechas válidas distribuidas en el tiempo.')

## 8. ¿Qué nos dicen realmente los datos?

- **Atención temática:** identifica las categorías más frecuentes en la tabla. Miden coincidencias con nuestro diccionario; no son una comparación exacta entre instituciones individuales. Para afirmar qué institución concreta recibe más atención habría que separar, por ejemplo, Congreso, Senado y agencias, y validar sus menciones.
- **Tono:** compara el porcentaje negativo dentro de cada categoría y también su tamaño. Una categoría pequeña puede tener un porcentaje alto por pocos mensajes. No interpretes los porcentajes como hostilidad dirigida a la institución.
- **Tiempo:** observa qué meses contienen datos, cambios de volumen y composición temática. Una diferencia puede reflejar selección, retuits o acontecimientos; no prueba una causa ni un cambio general del discurso.
- **Postura:** una crítica a una política, un insulto y una negación explícita de legitimidad son fenómenos diferentes. El mapa de sentimiento no permite distinguirlos; la anotación humana debe identificar un objetivo y una evidencia concreta.

La siguiente tabla resume las señales automáticas y recupera **ejemplos reales**, no seleccionados para confirmar una conclusión. Un único ejemplo no es representativo de toda la categoría. Busca también contraejemplos y revisa la categoría `Otros`.

In [ ]:
interpretacion = resumen_temas.join(
    porcentajes_categoria['Negativo'].rename('porcentaje_negativo_VADER'))
interpretacion['negativos_VADER'] = tabla_conjunta['Negativo']
display(interpretacion.sort_values('mensajes', ascending=False).round(2))

# Ejemplos reproducibles de cada combinación existente; no equivalen a etiquetas humanas.
for categoria in resumen_temas['mensajes'].nlargest(3).index:
    print('\n===', categoria, '===')
    for etiqueta in ORDEN_SENTIMIENTO:
        seleccion = df.loc[df['topics'].map(lambda ts: categoria in ts)
                           & df['sentiment_label'].eq(etiqueta)]
        if not seleccion.empty:
            registro = seleccion.sample(1, random_state=SEMILLA).iloc[0]
            print(f"[{etiqueta}, compound={registro['sentiment_score']:.3f}] ID {registro['id']}")
            print(registro['texto_legible'])
    print('Pregunta: ¿a quién se dirige la valoración y qué evidencia lo demuestra?')

## 9. Limitaciones metodológicas

- **Selección y representatividad:** no se documenta el muestreo y predomina «fake news». No es el conjunto completo de publicaciones ni una muestra de ciudadanos. El fichero mezcla contenidos y retuits; no permite comprobar por sí solo autoría, autenticidad o contexto.
- **Cobertura temporal:** faltan garantías de exhaustividad, cambian plataformas y periodos, y las fechas están redondeadas. No inferimos ritmos exactos de publicación.
- **Ambigüedad política:** un mensaje puede citar a otra persona, elogiar una institución y criticar otra, o referirse a un adversario sin hablar de una institución.
- **Ironía y sarcasmo:** VADER puede confundirlos. Su tono global no identifica el objetivo, no evalúa la veracidad ni determina postura o intención.
- **Palabras clave:** falsos positivos, omisiones, variantes no previstas y solapamientos. El diccionario y el criterio de empate afectan al resultado. Revisa una muestra dentro y fuera de cada categoría.
- **Correlación y causalidad:** coincidencias entre fechas, temas y tono no identifican relaciones causales. Los gráficos son descriptivos.
- **Crítica y democracia:** una crítica legítima no es un ataque a principios democráticos. Una etiqueta de deslegitimación requiere evidencia contextual y criterios compartidos, no solo términos negativos.
- **Validación humana:** antes de cuantificar posturas, necesita anotaciones independientes, revisión de desacuerdos y evaluación de fiabilidad. Para sentimiento, anota una muestra reservada y compara VADER con etiquetas humanas; para temas, revisa tanto coincidencias como omisiones. No uses sus propias predicciones como verdad de referencia.

**Reproducibilidad:** usamos una semilla fija, el léxico VADER 3.3.2 y mostramos versiones y revisión Git. Un clon futuro de `main` puede contener datos distintos; para repetir exactamente el estudio conserva esa revisión y las versiones registradas. No se exporta ni modifica el CSV original.

## 10. Actividad para los estudiantes

**Ejercicio 1 · Sensibilidad del diccionario**  
Añade palabras o expresiones a una categoría. Vuelve a ejecutar desde `compilar_taxonomia`, compara los recuentos y revisa cinco mensajes que hayan cambiado. ¿Mejora la cobertura o aparecen falsos positivos?

**Ejercicio 2 · Del sentimiento al objetivo**  
Selecciona cinco mensajes negativos y comprueba manualmente si critican una institución. Identifica el objetivo, copia una evidencia y distingue crítica, ataque y deslegitimación; deja pendientes los casos dudosos. Explica por qué el tono negativo no basta.

**Ejercicio 3 · Comparar categorías democráticas**  
Compara dos categorías: número de mensajes, proporción negativa y solapamiento. Incluye los denominadores y ejemplos, y explica cómo el diccionario y la selección del corpus afectan a la comparación.

**Ejercicio 4 · Diseñar una mejora**  
Propón embeddings, un clasificador supervisado o un LLM como ampliación conceptual (no se requiere ejecutarlo ni pagar servicios). Especifica datos de entrenamiento, anotación, separación de evaluación, métricas y errores que esperas reducir. ¿Cómo comprobarías que la mejora no introduce un nuevo sesgo?

**Preguntas para el debate**
- ¿Puede un mensaje ser negativo sin atacar a la democracia?
- ¿Puede un mensaje aparentemente positivo deslegitimar una institución?
- ¿Qué riesgos existen al utilizar modelos automáticos para estudiar el discurso político?
- ¿Qué evidencias necesitamos para afirmar que existe un patrón de deslegitimación institucional?

**Entrega sugerida:** cuatro gráficos comentados, la comparación de categorías, las cinco revisiones y una conclusión que distinga observaciones, interpretaciones y cuestiones no resueltas.